설계행렬 A는 절편 열과 특성 열을 포함한다. 제곱오차 합을 최소화하는 조건은 정규방정식 `AᵀA w = Aᵀy`다. 역행렬 표현에는 가역성 조건이 필요하다. 실제 계산은 역행렬을 명시적으로 만드는 대신 **A에 직접 최소제곱 풀이를 적용**한다.

오차가 독립이고 같은 분산 σ²을 갖는 평균 0의 정규분포라는 모형에서, 고정된 양의 σ에 대한 로그우도는 상수항과 제곱오차 항으로 나뉜다. 따라서 w에 대한 최대우도는 최소제곱과 연결된다. **실제 시계열 오차가 이 가정을 만족한다는 증명은 아니다.** CLT만으로 오차 가정을 정당화하지 않는다.


In [2]:
import numpy as np

In [ ]:
def fit_least_squares(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    b = np.ones([X.shape[0],1])
    A = np.concatenate([b,X], axis=1)
    u, s, vt = np.linalg.svd(A,full_matrices=False)
    c = u.T@y
    t = 2.22*1e-16 * max(X.shape[0], X.shape[1]+1) * np.max(s)
    mask = s>t
    z = np.zeros(c.shape)
    z[mask] = c[mask]/s[mask]
    beta = vt.T@z
    return beta
# 1. 정확한 직선 → 기대 계수: [1, 2]
X = np.array([[0], [1], [2]])
print(X.shape)
y = np.array([1, 3, 5])
print(fit_least_squares(X,y))
# 2. 음수 기울기 → 기대 계수: [3, -2]
X = np.array([[-2], [0], [2]])
y = np.array([7, 3, -1])
print(fit_least_squares(X,y))
# 3. 잡음 포함 → 기대 계수: 약 [0.666667, 2]
X = np.array([[0], [1], [2]])
y = np.array([1, 2, 5])
print(fit_least_squares(X,y))

(3, 1)
[1. 2.]
[ 3. -2.]
[0.66666667 2.        ]


In [4]:
def gaussian_log_likelihood(w: np.ndarray, X: np.ndarray, y: np.ndarray, sigma: float = 1.0) -> float:
    #검증
    if X.ndim != 2:
        raise ValueError
    if w.ndim != 1:
        raise ValueError
    if y.ndim != 1:
        raise ValueError
    if X.shape[0] != y.shape[0]:
        raise ValueError
    if w.shape[0] != X.shape[1]+1:
        raise ValueError
    if not (np.all(np.isfinite(X)) and np.all(np.isfinite(y)) and np.all(np.isfinite(w)) and np.all(np.isfinite(sigma))):
        raise ValueError
    if sigma <= 0:
        raise ValueError
    A = np.concatenate([np.ones([X.shape[0],1]),X], axis=1)
    l = -X.shape[0]*np.log(np.sqrt(2*np.pi)*sigma)-1/(2*sigma**2)*np.sum((y-A@w)**2)
    return l
# 1. 정확한 직선 → 기대 계수: [1, 2]
X = np.array([[0], [1], [2]])
y = np.array([1, 3, 5])
print(gaussian_log_likelihood(fit_least_squares(X,y),X,y))
# 2. 음수 기울기 → 기대 계수: [3, -2]
X = np.array([[-2], [0], [2]])
y = np.array([7, 3, -1])
print(gaussian_log_likelihood(fit_least_squares(X,y),X,y))
# 3. 잡음 포함 → 기대 계수: 약 [0.666667, 2]
X = np.array([[0], [1], [2]])
y = np.array([1, 2, 5])
print(gaussian_log_likelihood(fit_least_squares(X,y),X,y))

# 
X = np.array([[0], [1]])
y = np.array([1, 3])
print(gaussian_log_likelihood(np.array([1,2]),X,y))

-2.756815599614018
-2.756815599614018
-3.0901489329473515
-1.8378770664093453


In [5]:
# 1. x축 데이터 생성: 0부터 10까지 5개의 점 생성
x = np.linspace(0, 10, 1000)
np.random.shuffle(x)
x=x.reshape(1000,1)
# 2. 직선 방정식 설정 (기울기 m=2, y절편 b=3)
m = 2.1
b = 3
y = m * x + b
y = y.reshape(1000,)
print(fit_least_squares(x,y))
gaussian_log_likelihood(fit_least_squares(x,y),x,y)



[3.  2.1]


np.float64(-918.9385332046727)

In [6]:
from sklearn.linear_model import LinearRegression

lin = LinearRegression()
b = np.ones((1000,1))
a = np.concatenate([b,x],axis=1)
lin.fit(x,y)
print(lin.coef_[0], lin.intercept_)
use = np.array([lin.intercept_, lin.coef_[0]])
gaussian_log_likelihood(use,x,y)

2.100000000000001 2.9999999999999947


np.float64(-918.9385332046727)

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:

    #두 행렬의 길이가 같은지 확인
    if len(y_true) != len(y_pred):
        raise ValueError("배열의 길이가 다름")
    if len(y_true) < 1:
        raise ValueError("빈 배열")
    #MSE 계산
    mse = np.sum((y_true - y_pred)**2)/len(y_true)
    #MAE 계산
    mae = np.sum(abs(y_true - y_pred))/len(y_true)
    #R2 계산(n<2 또는 정답이 상수이면 NaN으로 표기)
    ssres = np.sum((y_true - y_pred)**2)
    sstot = np.sum((y_true - np.mean(y_true))**2)
    if sstot == 0 or len(y_true) < 2:
        r2 = np.nan
    else :
        r2 = 1 - ssres/sstot
    return {"mse":mse, "mae":mae, "r2":r2}

a = np.array([2,3,4])
b = np.array([1,3,5])
print(regression_metrics(a,b))



{'mse': np.float64(0.6666666666666666), 'mae': np.float64(0.6666666666666666), 'r2': np.float64(0.0)}


In [22]:
import pandas as pd

In [ ]:
df = pd.read_csv("output.csv", index_col=0)
df.index = pd.PeriodIndex(df.index, freq="Y")


,previous_value,target
1701,5.0,11.0
1702,11.0,16.0
1703,16.0,23.0
1704,23.0,36.0
1705,36.0,58.0
...,...,...
2004,63.7,40.4
2005,40.4,29.8
2006,29.8,15.2
2007,15.2,7.5


In [36]:
from sklearn.model_selection import train_test_split

In [68]:
idx_train = df.index[:199]
idx_validation = df.values[199:249]
idx_test = df.values[249:]
train = df.values[:199]
validation = df.values[199:249]
test = df.values[249:]
print(len(train), len(validation), len(test))
print(train.shape)
train


199 50 59
(199, 2)


array([[  5. ,  11. ],
       [ 11. ,  16. ],
       [ 16. ,  23. ],
       [ 23. ,  36. ],
       [ 36. ,  58. ],
       [ 58. ,  29. ],
       [ 29. ,  20. ],
       [ 20. ,  10. ],
       [ 10. ,   8. ],
       [  8. ,   3. ],
       [  3. ,   0. ],
       [  0. ,   0. ],
       [  0. ,   2. ],
       [  2. ,  11. ],
       [ 11. ,  27. ],
       [ 27. ,  47. ],
       [ 47. ,  63. ],
       [ 63. ,  60. ],
       [ 60. ,  39. ],
       [ 39. ,  28. ],
       [ 28. ,  26. ],
       [ 26. ,  22. ],
       [ 22. ,  11. ],
       [ 11. ,  21. ],
       [ 21. ,  40. ],
       [ 40. ,  78. ],
       [ 78. , 122. ],
       [122. , 103. ],
       [103. ,  73. ],
       [ 73. ,  47. ],
       [ 47. ,  35. ],
       [ 35. ,  11. ],
       [ 11. ,   5. ],
       [  5. ,  16. ],
       [ 16. ,  34. ],
       [ 34. ,  70. ],
       [ 70. ,  81. ],
       [ 81. , 111. ],
       [111. , 101. ],
       [101. ,  73. ],
       [ 73. ,  40. ],
       [ 40. ,  20. ],
       [ 20. ,  16. ],
       [ 16

In [72]:
coef = fit_least_squares(train[:,0].reshape(199,1), train[:,1])

In [73]:
y_validation_lin = coef[1]*validation[:,0] + coef[0]
y_validation_cons = validation[:,0]

In [75]:
print(regression_metrics(validation[:,1],y_validation_lin))
print(regression_metrics(validation[:,1],y_validation_cons))

{'mse': np.float64(467.5629663253631), 'mae': np.float64(16.960607095954675), 'r2': np.float64(0.682488654755609)}
{'mse': np.float64(485.20120000000003), 'mae': np.float64(17.316), 'r2': np.float64(0.6705109325981364)}


In [ ]:
valid = df.iloc[199:249]
valid["persistence_prediction"] = valid["previous_value"]
valid["linear_regression"] = valid["previous_value"]*coef[1]+coef[0]
valid["absolute_error_persistence_prediction"] = np.abs(valid["target"]-valid["persistence_prediction"])
valid["absolute_error_linear_regression"] = np.abs(valid["target"]-valid["linear_regression"])
print(valid.loc[valid["absolute_error_persistence_prediction"].idxmax()])
print(valid.loc[valid["absolute_error_linear_regression"].idxmax()])

previous_value                33.200000
target                        92.600000
persistence_prediction        33.200000
linear_regression             35.280854
mae_persistence_prediction    59.400000
mae_linear_regression         57.319146
Name: 1946, dtype: float64
previous_value                 92.600000
target                        151.600000
persistence_prediction         92.600000
linear_regression              83.721517
mae_persistence_prediction     59.000000
mae_linear_regression          67.878483
Name: 1947, dtype: float64
